In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

##Task 3.1

In [2]:
data = pd.read_csv("/content/appointments_clean.csv")
data.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [3]:
data.shape

(990, 16)

In [4]:
data.columns

Index(['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
       'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
       'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days',
       'no_show'],
      dtype='object')

In [5]:
x = data[['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent']]

In [6]:
x['reminder_sent'] = x['reminder_sent'].map({
    'Yes': 1,
    'No': 0
})

/tmp/ipykernel_710/49501536.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x['reminder_sent'] = x['reminder_sent'].map({


In [7]:
x['reminder_sent'].value_counts()

,count
reminder_sent,
1,561
0,429


In [8]:
x['booking_channel'] = data['booking_channel']
x['visit_type'] = data['visit_type']

/tmp/ipykernel_710/2628409833.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x['booking_channel'] = data['booking_channel']
/tmp/ipykernel_710/2628409833.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x['visit_type'] = data['visit_type']


In [9]:
x.head()

,age,distance_km,fee_inr,lead_days,prior_no_shows,reminder_sent,booking_channel,visit_type
0,60,18.0,850,1,1,1,Phone,Consultation
1,39,12.5,350,10,2,0,Walk-In,Follow-up
2,63,4.1,850,6,0,0,Walk-In,Consultation
3,52,7.5,750,4,0,1,App,Consultation
4,64,19.0,700,40,0,0,Phone,Consultation


In [10]:
x = pd.get_dummies(
    x,
    columns=['booking_channel', 'visit_type']
)

In [11]:
x.head()
x.shape

(990, 11)

In [12]:
y = data['no_show']

In [13]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [14]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((792, 11), (198, 11), (792,), (198,))

In [15]:
no_shows_test = (y_test == 1).sum()

print("No-Shows in test set:", no_shows_test)

No-Shows in test set: 49


In [16]:
total_test_appointment = y_test.shape

In [17]:
attended_appointment = total_test_appointment - no_shows_test
attended_appointment

array([149])

In [18]:
attended_appointment / total_test_appointment * 100

array([75.25252525])

##Task 3.2

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score
import numpy

In [20]:
scaled = StandardScaler()
scaled_x_train = scaled.fit_transform(x_train)
scaled_x_test = scaled.transform(x_test)

In [21]:
model = LogisticRegression(max_iter=1000)
model.fit(scaled_x_train,y_train)

LogisticRegression(max_iter=1000)

In [22]:
y_pred = model.predict(scaled_x_test)

In [23]:
print("accuracy_score:",accuracy_score(y_test,y_pred))

accuracy_score: 0.7878787878787878


In [24]:
print("confusion_matrix:",confusion_matrix(y_test,y_pred))

confusion_matrix: [[139  10]
 [ 32  17]]


In [25]:
print("precision_score:",precision_score(y_test,y_pred))

precision_score: 0.6296296296296297


In [26]:
print("recall_score:",recall_score(y_test,y_pred))

recall_score: 0.3469387755102041


In [27]:
print("f1_score:",f1_score(y_test,y_pred))

f1_score: 0.4473684210526316


In [28]:
y_prob = model.predict_proba(scaled_x_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.788933022873579


In [29]:
for feature, coefficient in zip(x.columns, model.coef_[0]):
    print(feature, ":", coefficient)

age : -0.2538801442716842
distance_km : 0.25344770956317536
fee_inr : 0.4527614630856424
lead_days : 0.6144707304969083
prior_no_shows : 0.6496116219621334
reminder_sent : -0.5806686033659922
booking_channel_App : -0.04680357040738989
booking_channel_Phone : 0.2158813749282703
booking_channel_Walk-In : -0.18636416836300682
visit_type_Consultation : 0.01565801179835534
visit_type_Follow-up : -0.015658011798355072


In [30]:
import numpy as np

reminder_coef = model.coef_[0][list(x.columns).index('reminder_sent')]

odds_ratio_scaled = np.exp(reminder_coef)

print("Scaled coefficient odds ratio:", odds_ratio_scaled)

Scaled coefficient odds ratio: 0.559524141750829


In [31]:
reminder_index = list(x.columns).index('reminder_sent')

reminder_std = scaled.scale_[reminder_index]

odds_ratio_reminder = np.exp(reminder_coef / reminder_std)

print("Reminder Yes vs No odds ratio:", odds_ratio_reminder)

Reminder Yes vs No odds ratio: 0.3093980903124433


In [32]:
print("Scaled coefficient OR:", odds_ratio_scaled)
print("Reminder Yes vs No OR:", odds_ratio_reminder)

Scaled coefficient OR: 0.559524141750829
Reminder Yes vs No OR: 0.3093980903124433


## Task 3.3


In [33]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [34]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(x_train, y_train)

train_pred = dt_model.predict(x_train)
test_pred = dt_model.predict(x_test)

print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

Training Accuracy: 1.0
Test Accuracy: 0.6717171717171717


In [35]:
dt_model_3 = DecisionTreeClassifier(max_depth=3, random_state=42)

dt_model_3.fit(x_train, y_train)

train_pred_3 = dt_model_3.predict(x_train)
test_pred_3 = dt_model_3.predict(x_test)

print("Training Accuracy:", accuracy_score(y_train, train_pred_3))
print("Test Accuracy:", accuracy_score(y_test, test_pred_3))

Training Accuracy: 0.821969696969697
Test Accuracy: 0.7777777777777778


In [36]:
dt_model_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt_model_3.fit(x_train, y_train)

train_pred_3 = dt_model_3.predict(x_train)
test_pred_3 = dt_model_3.predict(x_test)

print("Training Accuracy:", accuracy_score(y_train, train_pred_3))
print("Test Accuracy:", accuracy_score(y_test, test_pred_3))

Training Accuracy: 0.821969696969697
Test Accuracy: 0.7777777777777778


##Task 3.4

In [37]:
from sklearn.ensemble import RandomForestClassifier

In [39]:
rf = RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(x_train,y_train)

RandomForestClassifier(random_state=42)

In [40]:
rf_pred = rf.predict(x_test)

In [42]:
print('accuracy_score:',accuracy_score(y_test,rf_pred))
print("recall_score:",recall_score(y_test,rf_pred))

accuracy_score: 0.7474747474747475
recall_score: 0.30612244897959184


In [44]:
rf_prob = rf.predict_proba(x_test)[:,1]
print("rf_proba:",rf_prob)

rf_proba: [0.03 0.21 0.44 0.37 0.44 0.21 0.27 0.34 0.08 0.18 0.61 0.24 0.08 0.09
 0.38 0.04 0.14 0.42 0.07 0.73 0.21 0.13 0.17 0.56 0.79 0.22 0.11 0.56
 0.03 0.16 0.49 0.01 0.06 0.05 0.54 0.08 0.24 0.29 0.55 0.04 0.11 0.06
 0.08 0.04 0.08 0.76 0.02 0.63 0.01 0.07 0.01 0.16 0.19 0.18 0.56 0.62
 0.31 0.15 0.07 0.05 0.17 0.22 0.01 0.   0.53 0.06 0.61 0.11 0.68 0.56
 0.61 0.03 0.08 0.34 0.17 0.37 0.21 0.08 0.23 0.31 0.17 0.19 0.28 0.01
 0.24 0.1  0.06 0.16 0.07 0.19 0.01 0.   0.18 0.37 0.05 0.12 0.19 0.23
 0.08 0.02 0.1  0.14 0.2  0.22 0.24 0.16 0.57 0.04 0.19 0.23 0.09 0.04
 0.18 0.23 0.85 0.27 0.33 0.72 0.02 0.03 0.16 0.23 0.24 0.12 0.23 0.04
 0.02 0.16 0.39 0.28 0.11 0.56 0.21 0.23 0.5  0.45 0.07 0.2  0.12 0.19
 0.15 0.76 0.29 0.06 0.12 0.   0.01 0.   0.13 0.13 0.11 0.09 0.6  0.32
 0.26 0.44 0.51 0.51 0.64 0.06 0.9  0.47 0.26 0.05 0.03 0.04 0.01 0.11
 0.01 0.02 0.08 0.43 0.55 0.68 0.04 0.02 0.31 0.2  0.12 0.18 0.02 0.67
 0.56 0.5  0.02 0.03 0.06 0.51 0.16 0.41 0.38 0.01 0.2  0.37 0.36 0

In [45]:
print("roc_auc_score:",roc_auc_score(y_test,rf_prob))

roc_auc_score: 0.7057252431173813


In [47]:
importance = pd.DataFrame({
    "Feature": x.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance", ascending=False
)
importance.head(3)

,Feature,Importance
3,lead_days,0.219905
0,age,0.196788
1,distance_km,0.194739


In [48]:
recall_score(y_test, y_pred)

0.3469387755102041

In [52]:
print(roc_auc_score(y_test, y_prob))
print(recall_score(y_test, dt_model_3.predict(x_test)))
print(accuracy_score(y_test, y_pred))

0.788933022873579
0.22448979591836735
0.7878787878787878


In [53]:
dt_prob = dt_model_3.predict_proba(x_test)[:,1]
roc_auc_score(y_test,dt_prob)

np.float64(0.6996986714148747)

##Task 3.5


In [69]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score,cross_val_predict

In [56]:
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

In [57]:
rf_grid.fit(x_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 8], 'n_estimators': [100, 200]},
             scoring='roc_auc')

In [58]:
print("Best Parameters:", rf_grid.best_params_)

Best Parameters: {'max_depth': 3, 'n_estimators': 200}


In [60]:
print("Best CV ROC-AUC:", rf_grid.best_score_)

Best CV ROC-AUC: 0.7673242251739405


In [64]:
lr_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

In [65]:
lr_scores = cross_val_score(
    lr_pipeline,
    x_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)
print("Logistic Regression cv ROC-AUC:",lr_scores.mean())

Logistic Regression cv ROC-AUC: 0.7737139899948249


In [66]:
dt_scores = cross_val_score(
    dt_model_3,
    x_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)
print("Decision Tree CV ROC-AUC:",dt_scores.mean())

Decision Tree CV ROC-AUC: 0.6784494968661952


In [68]:
rf_model = cross_val_score(
    rf,
    x_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

print("Random Forest CV ROC-AUC",rf_model.mean())

Random Forest CV ROC-AUC 0.7466024782933701


In [72]:
oof_probabilities = cross_val_predict(
    lr_pipeline,
    x_train,
    y_train,
    cv=5,
    method='predict_proba'
)
oof_prob = oof_probabilities[:, 1]

In [73]:
for threshold in [0.50, 0.35]:

    oof_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(y_train, oof_pred)
    recall = recall_score(y_train, oof_pred)
    f1 = f1_score(y_train, oof_pred)
    flagged = oof_pred.sum()

    print("Threshold:", threshold)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("Number Flagged:", flagged)
    print("----------------------")

Threshold: 0.5
Precision: 0.5753424657534246
Recall: 0.2485207100591716
F1 Score: 0.34710743801652894
Number Flagged: 73
----------------------
Threshold: 0.35
Precision: 0.512987012987013
Recall: 0.46745562130177515
F1 Score: 0.4891640866873065
Number Flagged: 154
----------------------


In [74]:
lr_pipeline.fit(x_train,y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(max_iter=1000))])

In [75]:
test_probabilities = lr_pipeline.predict_proba(x_test)
test_prob = test_probabilities[:, 1]

In [76]:
final_threshold = 0.35
test_pred = (test_prob >= final_threshold).astype(int)

In [77]:
precision = precision_score(y_test, test_pred)
recall = recall_score(y_test, test_pred)
flagged = test_pred.sum()
auc = roc_auc_score(y_test, test_prob)

print("Precision:", precision)
print("Recall:", recall)
print("Number Flagged:", flagged)
print("ROC-AUC:", auc)

Precision: 0.5714285714285714
Recall: 0.5714285714285714
Number Flagged: 49
ROC-AUC: 0.788933022873579


In [79]:
x_test.index

Index([213, 331, 501, 309,  88, 535, 280, 107,  59, 506,
       ...
       448,  65, 141, 630, 745, 450, 705, 305, 807, 237],
      dtype='int64', length=198)

In [80]:
risk_scores = pd.DataFrame({
    'appointment_id': data.loc[x_test.index, 'appointment_id'],
    'no_show': y_test,
    'risk_prob': test_prob,
    'flag': test_pred
})

In [81]:
risk_scores.head()

,appointment_id,no_show,risk_prob,flag
213,A0217,0,0.021565,0
331,A0336,0,0.150917,0
501,A0508,0,0.518692,1
309,A0314,0,0.191300,0
88,A0089,0,0.167718,0


In [82]:
risk_scores.to_csv('test_risk_scores.csv', index=False)

In [83]:
risk_scores.shape

(198, 4)